In [1]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import gc

In [2]:
wine = fetch_openml(name='wine-quality-white', version=1, as_frame=False)
X, y = wine.data, wine.target.astype(int)
y = y - y.min()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
test_dataset  = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

train_loader = DataLoader(dataset=train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(dataset=test_dataset, batch_size=30, shuffle=False)

In [3]:
class mlp(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(11, 32),
            nn.ReLU(),
            nn.Linear(32, 7)
        )

    def forward(self, x):
        return self.layers(x)

model = mlp()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
gc.collect()

1060

In [4]:
epochs = 20
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

In [5]:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f'Epoch [{epoch + 1}/{epochs}]')
    print("Training loss", (running_loss / total))  
    train_losses.append(running_loss / total)
    print("Training accuracy", (correct *100/ total),'%')
    train_accuracies.append(correct * 100 / total)

    model.eval()
    running_loss_eval = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_eval += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Testing loss", (running_loss_eval / total))
    test_losses.append(running_loss_eval / total)
    print("Testing accuracy", (correct*100 / total),'%')
    test_accuracies.append(correct * 100 / total)
    
    gc.collect()

Epoch [1/20]
Training loss 1.173083482012571
Training accuracy 51.76110260336907 %
Testing loss 1.1119060127102598
Testing accuracy 53.06122448979592 %
Epoch [2/20]
Training loss 1.090574350510403
Training accuracy 54.74732006125574 %
Testing loss 1.1230216500710468
Testing accuracy 48.46938775510204 %
Epoch [3/20]
Training loss 1.0630343999711758
Training accuracy 55.15569167942828 %
Testing loss 1.0775221859922215
Testing accuracy 52.55102040816327 %
Epoch [4/20]
Training loss 1.053179922366763
Training accuracy 55.1301684532925 %
Testing loss 1.087030180254761
Testing accuracy 52.142857142857146 %
Epoch [5/20]
Training loss 1.0381637122814846
Training accuracy 55.81929555895865 %
Testing loss 1.0678518870655371
Testing accuracy 52.95918367346939 %
Epoch [6/20]
Training loss 1.0322967491446862
Training accuracy 55.742725880551305 %
Testing loss 1.0786937761063478
Testing accuracy 54.795918367346935 %
Epoch [7/20]
Training loss 1.0246720151246476
Training accuracy 55.87034201123022 %
